# Dev experiments: OSTrack + V-JEPA (Kaggle, self-contained)

Everything in one session, no notebook/dataset inputs (mounting large outputs fails on Kaggle).
Choose the steps in `RUN` (cell 4). The V-JEPA cache lives only in this session, so every
V-JEPA step needs `cache=True`.

- 0 - self-test on 30 frames (fails within minutes if something is broken) - ~5 min
- 1 - V-JEPA cache for the 40 dev videos (+ predictor surprise map if `surprise=True`, 2x time) - ~1 h
- 2 - OSTrack + V-JEPA re-localization, no verification - ~30 min
- 2b - re-localization verified by OSTrack (V-JEPA proposes 3 places) - ~30 min
- 2c - same, but OSTrack must be confident (score >= 0.6) at the accepted place - ~30 min
- 2d - tracking with V-JEPA alone, no OSTrack (sanity check) - ~5 min
- 2e - 2c + world model: move only after the V-JEPA predictor was surprised at the target - ~30 min
- 2f - 2c + YOLO-World detections of the target's class as extra candidates - ~35 min
- 3 - training samples from 280 train videos (+ samples with the target hidden) - ~1.5 h
- 4 - train fusion_v1 - ~30-60 min
- 4h - train fusion_v1h: fusion + OSTrack's score head, negatives, distillation - ~1 h
- 5 - OSTrack + fusion_v1 on dev - ~30 min
- 5h - OSTrack + fusion_v1h on dev: alone, + verified re-localization, + predictor surprise gate - ~1.5 h
- 4p - train fusion_v2a: V-JEPA target-memory prior in OSTrack's score head (~9.5k parameters) - ~1 h
- 5p - OSTrack + fusion_v2a on dev: alone, and + verified re-localization (score >= 0.6) - ~1 h

Results are packed into `/kaggle/working/dev_experiments.zip` after every step.

Settings: Accelerator = GPU T4 x2, Internet = On, no inputs needed. Then Save Version -> Save & Run All.

In [ ]:
# fail fast with a clear message if the session has no internet or no GPU
import socket
try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    raise SystemExit('NO INTERNET: Settings (right panel) -> Internet = On '
                     '(needs a phone-verified Kaggle account), then run again.')
import torch
if torch.cuda.device_count() == 0:
    raise SystemExit('NO GPU: Settings (right panel) -> Accelerator = GPU T4 x2, then run again.')
print(torch.cuda.device_count(), 'GPUs:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess  # check=True stops the run here if cloning fails
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/MariiaHamanuk/Object-Tracking.git',
                '/kaggle/working/Object-Tracking'], check=True)
!pip install -q timm==0.6.13 easydict visdom tensorboardX jpeg4py lmdb gdown "transformers>=4.53"
!pip install -q ultralytics "git+https://github.com/ultralytics/CLIP.git"
!gdown 1dySfPrSk-knAEo0lTXqZ1kMk3BCvVofV -O /kaggle/working/OSTrack_ep0300.pth.tar

In [ ]:
# which steps to run
RUN = dict(cache=True,
           surprise=False,       # predictor surprise in the cache (only for reloc_surprise / fusion_head)
           reloc=False,          # done: results_raw/ostrack_vjepa_reloc (AUC 55.58 on dev)
           reloc_verify=False,   # done: results_raw/ostrack_vjepa_reloc_verify (AUC 56.39 on dev)
           reloc_verify_min=False,  # done: results_raw/ostrack_vjepa_reloc_verify_min (AUC 57.61, best so far)
           vjepa_only=False,     # done: AUC 4.32
           reloc_surprise=False,  # done: AUC 56.07 (surprise does not detect occlusions)
           yolo=True,            # reloc_verify_min + YOLO-World detections as extra candidates
           warning=False,        # reloc_verify_min + occlusion warning (predictor surprise / score only); needs surprise=True
           prepare=True,         # training samples (needed by train / train_head / train_prior)
           train=False, fusion=False,   # fusion_v1 done: results_raw/ostrack_fusion_v1
           train_head=False, fusion_head=False,  # fusion_v1h done: AUC 57.11 / 57.37 / 57.44
           train_prior=True,     # fusion_v2a: V-JEPA target-memory prior in OSTrack's score head
           prior=True)           # track dev with fusion_v2a, alone and + verified re-localization
ZIP = 'dev_experiments.zip'

In [ ]:
import os, glob, subprocess, sys, torch, numpy as np
os.chdir('/kaggle/working/Object-Tracking')
sys.path.insert(0, '.')
os.makedirs('/kaggle/tmp', exist_ok=True)
W = '/kaggle/working'
CKPT = f'{W}/OSTrack_ep0300.pth.tar'
CACHE = '/kaggle/tmp/cache/vjepa'
DEV = 'configs/splits/dev_occlusion.txt'
N_GPU = max(torch.cuda.device_count(), 1)
os.makedirs(f'{W}/logs', exist_ok=True)

def run_shards(name, args):
    """Run a script once per GPU (each takes every N-th video), wait, show the log tail."""
    procs = []
    for i in range(N_GPU):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        log = open(f'{W}/logs/{name}_shard{i}.log', 'w')
        procs.append(subprocess.Popen(['python'] + args + ['--shard', str(i), '--num-shards', str(N_GPU)],
                                      env=env, stdout=log, stderr=subprocess.STDOUT))
    codes = [p.wait() for p in procs]
    os.system(f"grep -hE 'FAILED|Error|Traceback' {W}/logs/{name}_shard*.log | head -n 20")
    os.system(f"tail -n 2 {W}/logs/{name}_shard*.log")
    print(name, 'exit codes', codes)
    return codes

def pack():
    """Zip everything worth keeping (small files only) - called after every step."""
    os.system(f'cd {W} && rm -f {ZIP} && zip -qr {ZIP} results_raw logs output 2>/dev/null; ls -la {W}/{ZIP}')

## 0. Self-test (30 frames)
Plain OSTrack, re-localization and zero-initialised fusion must run; fusion before training must equal
plain OSTrack; a wrong candidate offered for verification must be rejected and leave the track unchanged.

In [ ]:
from src.tracker.fusion import FusionV1
os.makedirs('/kaggle/tmp/selftest/cache', exist_ok=True)
np.savez('/kaggle/tmp/selftest/cache/airplane-1.npz', frames=np.arange(0, 3000, 8),
         maps=np.random.randn(375, 16, 16, 1024).astype(np.float16),
         surprise=np.random.rand(375, 16, 16).astype(np.float16))
torch.save(FusionV1().state_dict(), '/kaggle/tmp/selftest/fusion_zero.pth')
base = ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', 'airplane-1', '--max-frames', '30',
        '--work', '/kaggle/tmp/selftest/work']
for name, extra in [('plain', []),
                    ('reloc', ['--vjepa-cache', '/kaggle/tmp/selftest/cache', '--reloc-verify', '3', '--verify-min', '0.6',
                               '--surprise-gate']),
                    ('warning', ['--vjepa-cache', '/kaggle/tmp/selftest/cache', '--reloc-verify', '3', '--verify-min', '0.6',
                                 '--occlusion-warning', 'both']),
                    ('fusion0', ['--vjepa-cache', '/kaggle/tmp/selftest/cache', '--fusion', '/kaggle/tmp/selftest/fusion_zero.pth'])]:
    subprocess.run(['python'] + base + ['--out', f'/kaggle/tmp/selftest/{name}'] + extra, check=True)
plain = np.loadtxt('/kaggle/tmp/selftest/plain/airplane-1.txt', delimiter='\t')
fusion0 = np.loadtxt('/kaggle/tmp/selftest/fusion0/airplane-1.txt', delimiter='\t')
print('max box difference, zero-init fusion vs plain OSTrack:', np.abs(plain - fusion0).max())
assert np.abs(plain - fusion0).max() < 1.0, 'zero-initialised fusion should reproduce OSTrack'
subprocess.run(['python', 'scripts/track_vjepa_only.py', '--sequences', 'airplane-1', '--max-frames', '30',
                '--vjepa-cache', '/kaggle/tmp/selftest/cache', '--out', '/kaggle/tmp/selftest/vjepa_only',
                '--work', '/kaggle/tmp/selftest/work'], check=True)
assert np.loadtxt('/kaggle/tmp/selftest/vjepa_only/airplane-1.txt', delimiter='\t').shape == (30, 4)
torch.save(dict(mode='AB', prior_head={}), '/kaggle/tmp/selftest/prior_zero.pth')   # all new layers = 0
subprocess.run(['python'] + base + ['--out', '/kaggle/tmp/selftest/prior0', '--vjepa-cache', '/kaggle/tmp/selftest/cache',
                '--prior', '/kaggle/tmp/selftest/prior_zero.pth'], check=True)
prior0 = np.loadtxt('/kaggle/tmp/selftest/prior0/airplane-1.txt', delimiter='\t')
print('max box difference, zero-init prior vs plain OSTrack:', np.abs(plain - prior0).max())
assert np.abs(plain - prior0).max() < 1.0, 'zero-initialised prior head should reproduce OSTrack'
if RUN['prepare'] or RUN['train_head'] or RUN['train_prior']:  # samples of 2 videos -> 1 epoch -> track
    subprocess.run(['python', 'scripts/prepare_fusion_data.py', '--max-videos', '2', '--per-video', '2',
                    '--out', '/kaggle/tmp/selftest/fdata', '--work', '/kaggle/tmp/selftest/work'], check=True)
    subprocess.run(['python', 'scripts/train_fusion.py', '--data', '/kaggle/tmp/selftest/fdata/motion', '--checkpoint', CKPT,
                    '--epochs', '1', '--batch', '4', '--prior', 'AB', '--negatives', '--lr', '1e-3',
                    '--out', '/kaggle/tmp/selftest/fv2a'], check=True)
    subprocess.run(['python'] + base + ['--out', '/kaggle/tmp/selftest/fv2a_track', '--vjepa-cache', '/kaggle/tmp/selftest/cache',
                    '--prior', '/kaggle/tmp/selftest/fv2a/prior.pth', '--reloc-verify', '3', '--verify-min', '0.6'], check=True)

from src.tracker.ostrack import OSTrackRunner
from src.data.remote_zip import open_class_zip, extract_frames
fdir = extract_frames(open_class_zip('airplane'), 'airplane-1', range(30), '/kaggle/tmp/selftest/frames')
frames = sorted((fdir / 'img').glob('*.jpg'))[:30]

if RUN['yolo']:  # YOLO-World finds the airplane in the first frame
    from src.tracker.detector import YOLODetector
    det = YOLODetector('yolov8s-worldv2.pt')
    det.set_class('airplane')
    import cv2
    boxes = det(cv2.cvtColor(cv2.imread(str(frames[0])), cv2.COLOR_BGR2RGB))
    from src.eval.metrics import iou
    gt0 = np.loadtxt(fdir / 'groundtruth.txt', delimiter=',')[:1]
    best = float(iou(boxes, np.repeat(gt0, len(boxes), 0)).max()) if len(boxes) else 0.0
    print('YOLO-World detections:', len(boxes), '| best IoU with the GT box:', round(best, 3))
    assert best > 0.3, 'YOLO-World does not find the target'
    del det
    subprocess.run(['python'] + base + ['--out', '/kaggle/tmp/selftest/yolo', '--vjepa-cache', '/kaggle/tmp/selftest/cache',
                    '--reloc-verify', '3', '--verify-min', '0.6', '--yolo', 'yolov8s-worldv2.pt'], check=True)

# V-JEPA predictor on real frames: a still clip must surprise less than one whose last frames are swapped
import cv2
from src.vjepa.predictor import VJEPAPredictor
rgb = [cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB) for f in frames[:16]]
pred = VJEPAPredictor()
_, s = pred.encode_with_surprise([rgb, rgb[:14] + [np.ascontiguousarray(rgb[0][::-1])] * 2])
print('predictor surprise, real clip vs last frames replaced by a flipped one:', float(s[0].mean()), float(s[1].mean()))
assert np.isfinite(s.float().numpy()).all() and s[1].float().mean() > s[0].float().mean(), 'predictor surprise looks wrong'
del pred
torch.cuda.empty_cache()
gt = np.loadtxt(fdir / 'groundtruth.txt', delimiter=',')
runner = OSTrackRunner(CKPT, save_dir='/kaggle/tmp/selftest/ost')
corner = lambda i, b, s: np.array([[0, 0, b[2], b[3]]]) if i % 10 == 5 else None   # a wrong place
res = runner.track(frames, gt[0], hook=corner, verify=True)
print('verification checks:', res['checks'])
assert len(res['checks']) == 3 and not any(c['accepted'] for c in res['checks']), 'wrong candidate was accepted'
print('max box difference, verified run vs plain OSTrack:', np.abs(res['boxes'] - plain).max())
assert np.abs(res['boxes'] - plain).max() < 1.0, 'rejected candidates must leave the track unchanged'
del runner
torch.cuda.empty_cache()
print('SELF-TEST OK')

## 1. V-JEPA cache for the dev videos

In [ ]:
if RUN['cache']:
    for attempt in range(2):  # a re-run only does the videos that failed to download
        run_shards('vjepa_cache', ['scripts/cache_vjepa.py', '--sequences', DEV, '--out', CACHE, '--work', '/kaggle/tmp']
                   + (['--surprise'] if RUN['surprise'] else []))
        n = len(glob.glob(f'{CACHE}/*.npz'))
        print(n, 'of 40 dev videos cached')
        if n == 40:
            break
    assert n == 40, 'V-JEPA cache incomplete'

## 2. OSTrack + V-JEPA re-localization on dev (no verification)

In [ ]:
if RUN['reloc']:
    run_shards('reloc', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV, '--vjepa-cache', CACHE,
                         '--out', f'{W}/results_raw/ostrack_vjepa_reloc/lasot', '--work', '/kaggle/tmp'])
    pack()

## 2b. Re-localization verified by OSTrack (V-JEPA proposes 3 places, OSTrack decides)

In [ ]:
if RUN['reloc_verify']:
    run_shards('reloc_verify', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV,
                                '--vjepa-cache', CACHE, '--reloc-verify', '3',
                                '--out', f'{W}/results_raw/ostrack_vjepa_reloc_verify/lasot', '--work', '/kaggle/tmp'])
    pack()

## 2c. Verified re-localization, OSTrack must be confident (score >= 0.6) at the accepted place

In [ ]:
if RUN['reloc_verify_min']:
    run_shards('reloc_verify_min', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV,
                                    '--vjepa-cache', CACHE, '--reloc-verify', '3', '--verify-min', '0.6',
                                    '--out', f'{W}/results_raw/ostrack_vjepa_reloc_verify_min/lasot', '--work', '/kaggle/tmp'])
    pack()

## 2d. Tracking with V-JEPA alone (no OSTrack)

In [ ]:
if RUN['vjepa_only']:
    subprocess.run(['python', 'scripts/track_vjepa_only.py', '--sequences', DEV, '--vjepa-cache', CACHE,
                    '--out', f'{W}/results_raw/vjepa_only/lasot', '--work', '/kaggle/tmp'], check=True)
    pack()

## 2e. + world model: move only after the V-JEPA predictor was surprised at the target
Same as 2c, but OSTrack's doubt only counts if the predictor's surprise at the last confident box was
high within the last 64 frames (the target vanished in a way the model did not expect).
`<seq>_reloc.csv` logs the surprise at every check, to see how well it detects occlusions.

In [ ]:
if RUN['reloc_surprise']:
    run_shards('reloc_surprise', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV,
                                  '--vjepa-cache', CACHE, '--reloc-verify', '3', '--verify-min', '0.6', '--surprise-gate',
                                  '--out', f'{W}/results_raw/ostrack_vjepa_reloc_surprise/lasot', '--work', '/kaggle/tmp'])
    pack()

## 2f. 2c + YOLO-World: detections of the target's class as extra candidates
Same trigger as 2c; when it fires, YOLO-World (prompted with the LaSOT class name) detects objects in the
frame, up to 3 outside OSTrack's window are added to V-JEPA's 3 candidates (ranked by V-JEPA similarity to
the target memory), and OSTrack verifies all of them (score >= 0.6). `<seq>_verify.csv` -> best_index
>= 3 means a YOLO box won.

In [ ]:
if RUN['yolo']:
    run_shards('reloc_yolo', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV,
                              '--vjepa-cache', CACHE, '--reloc-verify', '3', '--verify-min', '0.6',
                              '--yolo', 'yolov8s-worldv2.pt',
                              '--out', f'{W}/results_raw/ostrack_vjepa_reloc_yolo/lasot', '--work', '/kaggle/tmp'])
    pack()

## 2g. 2c + occlusion warning from the V-JEPA predictor (and the same with the score only, as a control)
The predictor's surprise at the target is high when it starts to go behind something (dev: AUROC 0.68
right before a full occlusion, nearly independent of OSTrack's score; together 0.72). Used as an early
warning: memory, last box and statistics are frozen while warned, and moves are easier (OSTrack still
verifies them, score >= 0.6). `--occlusion-warning score` uses only OSTrack's score drop: the difference
between the two runs is what the predictor adds.

In [ ]:
if RUN['warning']:
    for mode in ('both', 'score'):
        run_shards(f'warning_{mode}', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV,
                                       '--vjepa-cache', CACHE, '--reloc-verify', '3', '--verify-min', '0.6',
                                       '--occlusion-warning', mode,
                                       '--out', f'{W}/results_raw/ostrack_vjepa_reloc_warning_{mode}/lasot',
                                       '--work', '/kaggle/tmp'])
        pack()

## 3. Training samples (280 train videos, 4 per class)

In [ ]:
if RUN['prepare']:
    run_shards('prepare', ['scripts/prepare_fusion_data.py', '--out', '/kaggle/tmp/fusion_data', '--work', '/kaggle/tmp'])
    print(len(glob.glob('/kaggle/tmp/fusion_data/motion/*.npz')), 'videos with samples')

## 4. Train fusion_v1 (one GPU)

In [ ]:
if RUN['train']:
    subprocess.run(['python', 'scripts/train_fusion.py', '--data', '/kaggle/tmp/fusion_data/motion', '--checkpoint', CKPT,
                    '--out', f'{W}/output/fusion_v1'], check=True)
    pack()

## 4h. Train fusion_v1h (fusion + OSTrack's score head, negatives, distillation; one GPU)

In [ ]:
if RUN['train_head']:
    subprocess.run(['python', 'scripts/train_fusion.py', '--data', '/kaggle/tmp/fusion_data/motion', '--checkpoint', CKPT,
                    '--train-head', 'score', '--negatives', '--out', f'{W}/output/fusion_v1h'], check=True)
    pack()

## 5h. OSTrack + fusion_v1h on dev: alone, and with verified re-localization (score >= 0.6)

In [ ]:
if RUN['fusion_head']:
    FH = ['--vjepa-cache', CACHE, '--fusion', f'{W}/output/fusion_v1h/fusion.pth',
          '--fusion-head', f'{W}/output/fusion_v1h/head.pth']
    run_shards('fusion_head', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV] + FH +
               ['--out', f'{W}/results_raw/ostrack_fusion_v1h/lasot', '--work', '/kaggle/tmp'])
    pack()
    run_shards('fusion_head_reloc', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV] + FH +
               ['--reloc-verify', '3', '--verify-min', '0.6',
                '--out', f'{W}/results_raw/ostrack_fusion_v1h_reloc_verify/lasot', '--work', '/kaggle/tmp'])
    pack()
    run_shards('fusion_head_surprise', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV] + FH +
               ['--reloc-verify', '3', '--verify-min', '0.6', '--surprise-gate',
                '--out', f'{W}/results_raw/ostrack_fusion_v1h_reloc_surprise/lasot', '--work', '/kaggle/tmp'])
    pack()

## 4p. Train fusion_v2a: V-JEPA target-memory prior in OSTrack's score head (one GPU)
OSTrack stays frozen; its score branch also reads 4 numbers per search token (similarity to the first /
recent target appearance in V-JEPA space, and whether something else in the frame looks more like the
target). ~9.5k new zero-initialised parameters, trained with negatives (target not in the crop).

In [ ]:
if RUN['train_prior']:
    subprocess.run(['python', 'scripts/train_fusion.py', '--data', '/kaggle/tmp/fusion_data/motion', '--checkpoint', CKPT,
                    '--prior', 'AB', '--negatives', '--lr', '1e-3', '--out', f'{W}/output/fusion_v2a'], check=True)
    pack()

## 5p. OSTrack + fusion_v2a on dev: alone, and + verified re-localization (score >= 0.6)

In [ ]:
if RUN['prior']:
    PR = ['--vjepa-cache', CACHE, '--prior', f'{W}/output/fusion_v2a/prior.pth']
    run_shards('prior', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV] + PR +
               ['--out', f'{W}/results_raw/ostrack_fusion_v2a/lasot', '--work', '/kaggle/tmp'])
    pack()
    run_shards('prior_reloc', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV] + PR +
               ['--reloc-verify', '3', '--verify-min', '0.6',
                '--out', f'{W}/results_raw/ostrack_fusion_v2a_reloc_verify/lasot', '--work', '/kaggle/tmp'])
    pack()

## 5. OSTrack + fusion_v1 on dev

In [ ]:
if RUN['fusion']:
    run_shards('fusion', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV, '--vjepa-cache', CACHE,
                          '--fusion', f'{W}/output/fusion_v1/fusion.pth',
                          '--out', f'{W}/results_raw/ostrack_fusion_v1/lasot', '--work', '/kaggle/tmp'])
    pack()

In [ ]:
pack()
os.system(f'rm -rf {W}/Object-Tracking {CKPT}; du -sh {W}/*')